In [ ]:
!pip install transformers biopython
from transformers import AutoTokenizer, AutoModel
import torch
model_name = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [ ]:
from google.colab import files
uploaded = files.upload()
with open('sequences.fasta', 'r') as f:
  sequences = f.read()
sequences = sequences.splitlines()

In [ ]:
# CREATE EMBEDDINGS DICTIONARY BY PROTEIN

embeddings_dict = {}
sequences_dict = {}

i = 0
while len(embeddings_dict) <= len(sequences)/2:
  if sequences[i].startswith(">"):
    protein_name = sequences[i].split(">")[1].strip()
    seq = sequences[i+1]    # next line is the actual sequence
    sequences_dict[protein_name] = seq   # add it to list of proteins we found embeddings for

    inputs = tokenizer(seq, return_tensors="pt")
    with torch.no_grad():
      outputs = model(**inputs)
    output2 = outputs.last_hidden_state
    output2 = output2[:, 1:-1, :]

    embeddings_dict[protein_name] = output2[0]

  if i % 100 == 0:
    print(i)

  i = i + 1


In [ ]:
import pickle
f = open("save_all.pkl","wb")
pickle.dump(embeddings_dict,f)
f.close()

print(embeddings_dict)

In [ ]:
# TO LOAD EMBEDDINGS DICITONARY
with open('save_all.pkl', 'rb') as file:
    # Load the data from the pickle file
    embeddings_dict = pickle.load(file)


In [ ]:
import pandas as pd
import numpy as np
import re

cols = 320
columns = [f'col{i+1}' for i in range(320)]
label_map = {'H':0, 'B':1, 'E':2, 'G':3, 'I':4, 'P':5, 'T':6, 'S':7, '.':8}

In [ ]:
# CREATE 6 TRAINING SETS (BATCHES)
from google.colab import files
uploaded = files.upload()
with open('trainF.tsv', 'r') as f:
  train = f.read()
train = train.splitlines()

rows = len(train)

In [ ]:
# DONE: A B C D
# E F- downloading
# working on:
train_embeddings = [torch.tensor([0 for i in range(320)])]
train_labels = []

i = 0
for line in train:
  splitted = re.split(r'[\t_]+', line)   # ['3L1W', 'LYS', '2', 'E']
  protein = splitted[0]
  position = int(splitted[2])
  label = splitted[3]

  if protein in embeddings_dict:
    if position <= embeddings_dict[protein].shape[0]:   # check if position is valid
      train_embeddings.append(embeddings_dict[protein][position-1])
      train_labels.append(label_map[label])

  if i % 150000 == 0:
    print(i)

  i = i+1

train_embeddings = torch.stack(train_embeddings)
train_embeddings = pd.DataFrame(train_embeddings.numpy())
train_embeddings = train_embeddings[1:]

train_labels = pd.DataFrame(train_labels)

In [ ]:
train_embeddings.to_pickle('trainA_embeddings.pkl')
train_labels.to_pickle('trainA_labels.pkl')
from google.colab import files
files.download("trainA_embeddings.pkl")
files.download("trainA_labels.pkl")

In [ ]:
train_embeddings.to_pickle('trainB_embeddings.pkl')
train_labels.to_pickle('trainB_labels.pkl')
from google.colab import files
files.download("trainB_embeddings.pkl")
files.download("trainB_labels.pkl")

In [ ]:
train_embeddings.to_pickle('trainC_embeddings.pkl')
train_labels.to_pickle('trainC_labels.pkl')
from google.colab import files
files.download("trainC_embeddings.pkl")
files.download("trainC_labels.pkl")

In [ ]:
train_embeddings.to_pickle('trainD_embeddings.pkl')
train_labels.to_pickle('trainD_labels.pkl')
from google.colab import files
files.download("trainD_embeddings.pkl")
files.download("trainD_labels.pkl")

In [ ]:
train_embeddings.to_pickle('trainE_embeddings.pkl')
train_labels.to_pickle('trainE_labels.pkl')
from google.colab import files
files.download("trainE_embeddings.pkl")
files.download("trainE_labels.pkl")

In [ ]:
train_embeddings.to_pickle('trainF_embeddings.pkl')
train_labels.to_pickle('trainF_labels.pkl')
from google.colab import files
files.download("trainF_embeddings.pkl")
files.download("trainF_labels.pkl")

In [ ]:
with open('test_noheader.tsv', 'r') as f:
  test = f.read()
test = test.splitlines()

In [ ]:
# GET EMBEDDINGS FOR TEST DATA
import re
import torch
import pandas as pd

test_embeddings = [torch.tensor([0 for i in range(320)])]
missing = []

index = 1
for line in test:
  splitted = re.split(r'[\t_]+', line)   # ['3L1W', 'LYS', '2']
  protein = splitted[0]
  position = int(splitted[2])

  if protein in embeddings_dict:
    if position <= embeddings_dict[protein].shape[0]:   # check if position is valid
      test_embeddings.append(embeddings_dict[protein][position-1])
    else:
      missing.append(index)
  else:
    missing.append(index)

  if index % 150000 == 0:
    print(index)

  index = index+1

test_embeddings = torch.stack(test_embeddings)
test_embeddings = pd.DataFrame(test_embeddings.numpy())
test_embeddings = test_embeddings[1:]     # gets rid of the header, i think

missing = pd.DataFrame(missing)

In [ ]:
# SAVE TEST EMBEDDINGS, SAVE MISSING

test_embeddings.to_pickle('test_embeddings.pkl')
missing.to_pickle('missing.pkl')
from google.colab import files
files.download("test_embeddings.pkl")
files.download("missing.pkl")

In [ ]:
# NOW WE TRAIN
import pickle
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

%cd /content/drive/MyDrive/Colab\ Notebooks/Project1_data

In [ ]:
# TO LOAD EMBEDDINGS DICITONARY
with open('save_all.pkl', 'rb') as file:
    # Load the data from the pickle file
    embeddings_dict = pickle.load(file)


In [ ]:
n_estimators = 50
max_features = 'sqrt'
max_depth = 10
n_jobs=-1
models = []

In [ ]:
# A
with open('trainA_embeddings.pkl', 'rb') as file:
    trainA_embeddings = pickle.load(file)
with open('trainA_labels.pkl', 'rb') as file:
    trainA_labels = pickle.load(file)
trainA_labels = trainA_labels.values.flatten()

clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth = max_depth, random_state=42, n_jobs=n_jobs)
clf.fit(trainA_embeddings, trainA_labels)
models.append(clf)

del trainA_embeddings
del trainA_labels

# B
with open('trainB_embeddings.pkl', 'rb') as file:
    trainB_embeddings = pickle.load(file)
with open('trainB_labels.pkl', 'rb') as file:
    trainB_labels = pickle.load(file)
trainB_labels = trainB_labels.values.flatten()

clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth = max_depth, random_state=42, n_jobs=n_jobs)
clf.fit(trainB_embeddings, trainB_labels)
models.append(clf)

del trainB_embeddings
del trainB_labels

# C
with open('trainC_embeddings.pkl', 'rb') as file:
    trainC_embeddings = pickle.load(file)
with open('trainC_labels.pkl', 'rb') as file:
    trainC_labels = pickle.load(file)
trainC_labels = trainC_labels.values.flatten()

clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth = max_depth, random_state=42, n_jobs=n_jobs)
clf.fit(trainC_embeddings, trainC_labels)
models.append(clf)

del trainC_embeddings
del trainC_labels

# D
with open('trainD_embeddings.pkl', 'rb') as file:
    trainD_embeddings = pickle.load(file)
with open('trainD_labels.pkl', 'rb') as file:
    trainD_labels = pickle.load(file)
trainD_labels = trainD_labels.values.flatten()

clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth = max_depth, random_state=42, n_jobs=n_jobs)
clf.fit(trainD_embeddings, trainD_labels)
models.append(clf)

del trainD_embeddings
del trainD_labels


# E
with open('trainE_embeddings.pkl', 'rb') as file:
    trainE_embeddings = pickle.load(file)
with open('trainE_labels.pkl', 'rb') as file:
    trainE_labels = pickle.load(file)
trainE_labels = trainE_labels.values.flatten()

clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth = max_depth, random_state=42, n_jobs=n_jobs)
clf.fit(trainE_embeddings, trainE_labels)
models.append(clf)

del trainE_embeddings
del trainE_labels

# F
with open('trainF_embeddings.pkl', 'rb') as file:
    trainF_embeddings = pickle.load(file)
with open('trainF_labels.pkl', 'rb') as file:
    trainF_labels = pickle.load(file)
trainF_labels = trainF_labels.values.flatten()

clf = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth = max_depth, random_state=42, n_jobs=n_jobs)
clf.fit(trainF_embeddings, trainF_labels)
models.append(clf)

del trainF_embeddings
del trainF_labels

In [ ]:
# SAVE MODEL
import pickle
i = 1
for model_i in models:
  with open(f"model{i}.pkl", "wb") as f:
    pickle.dump(model_i, f)
  i = i + 1

In [ ]:
# LOAD MODEL
# TBC
import pickle
models = []
with open('model1.pkl', 'rb') as file:
    # Load the data from the pickle file
    models.append(pickle.load(file))
with open('model2.pkl', 'rb') as file:
    # Load the data from the pickle file
    models.append(pickle.load(file))
with open('model3.pkl', 'rb') as file:
    # Load the data from the pickle file
    models.append(pickle.load(file))
with open('model4.pkl', 'rb') as file:
    # Load the data from the pickle file
    models.append(pickle.load(file))
with open('model5.pkl', 'rb') as file:
    # Load the data from the pickle file
    models.append(pickle.load(file))
with open('model6.pkl', 'rb') as file:
    # Load the data from the pickle file
    models.append(pickle.load(file))



In [ ]:
# Load TEST EMBEDDINGS, MISSING
with open('test_embeddings.pkl', 'rb') as file:
    test_embeddings = pickle.load(file)
with open('missing.pkl', 'rb') as file:
    missing = pickle.load(file)
missing = missing.values.flatten()

In [ ]:
# MAKE PREDICTIONS
import numpy as np
preds = np.array([model.predict(test_embeddings) for model in models])
final_preds = np.round(preds.mean(axis=0)).astype(int)

In [ ]:
# MAP, AND MAKE UP FOR MISSING VALUES
label_inverse_map = {0:'H', 1:'B', 2:'E', 3:'G', 4:'I', 5:'P', 6:'T', 7:'S', 8:'.'}
final_preds_mapped = []

true_index = 1
index = 1
while true_index <= 674154:
  if true_index in missing:
    final_preds_mapped.append('.')
  else:
    final_preds_mapped.append(label_inverse_map[final_preds[index-1]])
    index = index + 1

  true_index = true_index + 1


In [ ]:
# SAVE AS PREDICTIONS
import pandas as pd

output_df = pd.DataFrame({
    'id' : test,
    'prediction' : final_preds_mapped})

output_df.to_csv("predictions2.csv", sep='\t', index=False)

from google.colab import files
files.download("predictions2.csv")